# Example 5.8

In [ ]:
# Script to solve the laminar flow development problem
# in a square-duct using the ADI method

import numpy as np
import matplotlib.pyplot as plt


def Flow_ADI(L, H, N, M, K, dpdz, rho, mu, tf):
    """
    Solves the transient flow in a rectangular duct using the ADI method.

    Parameters
    ----------
    L, H : float
        Length and height of the cross section (cm)
    N, M, K : int
        Number of discrete segments in x, y, and time directions
    dpdz : float
        Axial pressure gradient (dyn/cm^3)
    rho, mu : float
        Fluid density (g/cm^3) and viscosity (g/cm/s)
    tf : float
        Final integration time (s)

    Returns
    -------
    V : ndarray
        Velocity distributions at different times (third dimension)
    VC : ndarray
        Centerline velocity as a function of time (length K+1)
    """

    # Numerical model parameters
    dx = L / N
    dy = H / M
    dt = tf / K

    gamma = 2 * dx**2 / (mu * dt / rho)
    g = dx**2 * dpdz / mu

    # Velocity arrays
    V = np.zeros((N + 1, M + 1, K + 1))
    Vint = np.zeros((N + 1, M + 1))

    # Tridiagonal matrix for implicit solves
    A = np.zeros((N + 1, N + 1))
    b = np.zeros(N + 1)

    for i in range(1, N):
        A[i, i - 1:i + 2] = [1, -(2 + gamma), 1]

    A[0, 0] = 1
    A[N, N] = 1

    # ADI (IDA) time-stepping loop
    for k in range(K):

        # First half-step (implicit in x)
        for j in range(1, M):
            for i in range(1, N):
                b[i] = (
                    g
                    - V[i, j - 1, k]
                    - V[i, j + 1, k]
                    + (2 - gamma) * V[i, j, k]
                )
            v = np.linalg.solve(A, b)
            Vint[:, j] = v

        # Second half-step (implicit in y)
        for i in range(1, N):
            for j in range(1, M):
                b[j] = (
                    g
                    - Vint[i - 1, j]
                    - Vint[i + 1, j]
                    + (2 - gamma) * Vint[i, j]
                )
            v = np.linalg.solve(A, b)
            V[i, :, k + 1] = v

    # Coordinates
    x = np.linspace(0, L, N + 1)
    y = np.linspace(0, H, M + 1)
    T = np.linspace(0, tf, K + 1)

    # Plot results
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.contourf(x, y, V[:, :, -1].T, cmap="jet")
    plt.xlabel('x, cm')
    plt.ylabel('y, cm')
    plt.colorbar(label='velocity, cm/s')

    # Centerline velocity
    i1 = N // 2
    i2 = M // 2
    VC = V[i1, i2, :]

    plt.subplot(1, 2, 2)
    plt.plot(T, VC, '-ro')
    plt.xlabel('time, s')
    plt.ylabel('centerline velocity, cm/s')

    plt.tight_layout()
    filename = 'Figure_5_11.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

    return V, VC


In [ ]:


N = 30        # number of nodes in the x direction
M = 30        # number of nodes in the y direction
L = 3.0       # duct length (cm)
H = 3.0       # duct height (cm)

rho = 1.0     # fluid density (g/cm^3)
mu = 0.03     # fluid viscosity (g/cm/s)
dpdz = -0.7   # axial pressure gradient (dyn/cm^3)

tf = 100.0    # final integration time (s)
K = 10        # number of time steps


V, Vc = Flow_ADI(L, H, N, M, K, dpdz, rho, mu, tf)